In [6]:
import pandas as pd
import os
import random
from dotenv import load_dotenv
from openai import OpenAI



DATA_PATH='../../data/'

CONTEXT_REL_CONFIG = {
    "name": "gpt-5.6-luna",
    "reasoning_effort": "low",
}

load_dotenv(override=True)

model_config = CONTEXT_REL_CONFIG

MODEL_NAME = model_config["name"]
REASONING_EFFORT = model_config["reasoning_effort"]

print(f"Using model: {MODEL_NAME} with reasoning effort: {REASONING_EFFORT}")


OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

client = OpenAI(
    api_key=OPENAI_API_KEY
)

df=pd.read_csv(f'{DATA_PATH}df_n1.csv')



Using model: gpt-5.6-luna with reasoning effort: low


code hunks:

In [10]:
import ast
import json
import random
import re
import time

def _parse_hunks_field(value):
    """
    pr_code_hunks / same_file_code_hunks are expected to be a list of
    {"filename": ..., "patch": ...} dicts. If the dataframe was loaded
    from CSV they may have come back in as stringified lists, so this
    normalizes both cases to an actual Python list.
    """
    if value is None:
        return []
    if isinstance(value, float) and pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    if isinstance(value, list):
        return value
    return []


# ==========================
# Candidate hunks (the ones the model gets to filter)
# ==========================

def get_same_file_candidates(row):
    """
    Other code hunks touched in the SAME target_file, in the same PR,
    excluding hunk x itself. Matching against `old_hunk`, which holds
    the raw patch text for hunk x as it appears inside
    pr_code_hunks / same_file_code_hunks. Each candidate gets a local id
    (1..N) used to reference it in the model's JSON output.
    """
    hunks = _parse_hunks_field(row.get("same_file_code_hunks"))
    current_patch = row.get("hunk")
    current_patch = str(current_patch).strip() if current_patch is not None else None

    candidates = []
    next_id = 1
    for h in hunks:
        if not isinstance(h, dict):
            continue
        patch = h.get("patch")
        if patch is None:
            continue
        if current_patch is not None and patch.strip() == current_patch:
            continue  # this is hunk x itself, skip it
        candidates.append(
            {
                "id": next_id,
                "filename": h.get("filename", row.get("target_file")),
                "patch": patch,
            }
        )
        next_id += 1
    return candidates


def get_other_file_candidates(row):
    """
    Code hunks touched in OTHER files (not target_file) within the same PR.
    Each candidate gets a local id (1..N), independent from the
    same-file candidate ids.
    """
    hunks = _parse_hunks_field(row.get("pr_code_hunks"))
    target_file = row.get("target_file")

    candidates = []
    next_id = 1
    for h in hunks:
        if not isinstance(h, dict):
            continue
        filename = h.get("filename")
        patch = h.get("patch")
        if filename is None or patch is None:
            continue
        if filename == target_file:
            continue
        candidates.append({"id": next_id, "filename": filename, "patch": patch})
        next_id += 1

    return candidates


def format_candidates(candidates, label):
    """label is 'SF' for same-file candidates or 'OF' for other-file candidates."""
    if not candidates:
        return ""
    blocks = []
    for c in candidates:
        header = f"[{label}-{c['id']}]"
        if label == "OF":
            header += f" file: {c['filename']}"
        # blocks.append(f"{header}\n{_truncate(c['patch'])}")
        blocks.append(f"{header}\n{c['patch']}")
    return "\n\n".join(blocks)


# ==========================
# Prompt construction
# ==========================

def build_relevance_messages(row, same_file_candidates, other_file_candidates):

    system_prompt = """
You are an expert software engineer helping to prepare context for an automated code review comment generator.

You will be given a target code change, called the "target hunk" (<hunk>), that a review comment will be generated for. You will also be given candidate code hunks that were changed elsewhere in the same pull request:

- <same_file_hunks>: other hunks changed in the SAME file as the target hunk.
- <other_file_hunks>: hunks changed in DIFFERENT files, in the same pull request.

Each candidate hunk has a numeric id shown in brackets (e.g. [SF-2], [OF-1]). The two lists use independent numbering — ids restart at 1 in each list. Other-file candidates also show the filename they belong to.

Your task:
For EACH list independently, select ONLY the candidate hunks that are relevant to understanding, evaluating, or reviewing the target hunk. A candidate hunk is relevant if knowing about it would change or inform a reviewer's comment about the target hunk — for example: it touches the same function, class, or variable; it is part of the same logical change (a rename, a signature change, a refactor the target hunk depends on or was caused by); or it establishes behavior/context the target hunk relies on.

A candidate hunk is IRRELEVANT if it is none of the above.

Be selective. The goal is to REDUCE the number of hunks passed downstream to only what a reviewer would actually need to correctly review the target hunk. When a candidate hunk's connection to the target hunk is unclear or weak, exclude it.

Output requirements:
Respond with ONLY a single JSON object and nothing else — don't chnage the code hunks, no explanations, no markdown fences, no extra text. The JSON must have exactly this shape:

{"relevant_same_file_ids": [<ints>], "relevant_other_files_ids": [<ints>]}

Use an empty array for a list if none of its candidates are relevant, or if that list was not shown to you. Never invent ids that were not shown to you.
"""

    user_prompt = f"""
<hunk>
{row["hunk"]}
</hunk>
"""

    same_file_block = format_candidates(same_file_candidates, "SF")
    other_file_block = format_candidates(other_file_candidates, "OF")

    if same_file_block:
        user_prompt += f"""
<same_file_hunks>
{same_file_block}
</same_file_hunks>
"""

    if other_file_block:
        user_prompt += f"""
<other_file_hunks>
{other_file_block}
</other_file_hunks>
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


# ==========================
# Output parsing
# ==========================

def parse_relevance_output(text):
    """
    Expects a JSON object like:
    {"relevant_same_file_ids": [1, 3], "relevant_other_files_ids": [2]}
    Falls back to empty lists on any parsing failure.
    """
    empty = {"relevant_same_file_ids": [], "relevant_other_files_ids": []}

    if text is None:
        return empty

    cleaned = text.strip()
    # strip markdown fences if the model added them despite instructions
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        print("[WARN] Could not parse relevance JSON output")
        return empty

    def _as_int_list(value):
        if not isinstance(value, list):
            return []
        result = []
        for v in value:
            try:
                result.append(int(v))
            except (TypeError, ValueError):
                continue
        return result

    return {
        "relevant_same_file_ids": _as_int_list(data.get("relevant_same_file_ids", [])),
        "relevant_other_files_ids": _as_int_list(data.get("relevant_other_files_ids", [])),
    }





In [11]:
import json
import os
import pandas as pd


def build_context_relevance_batch_jsonl(
    df,
    output_path="code_hunks_relevance_batch.jsonl",
):
    """
    Build a JSONL file containing one Batch API request per target hunk.

    Each line corresponds to one target hunk that has candidate hunks
    available for relevance filtering.
    """

    if "hunk" not in df.columns:
        raise ValueError("Missing required column: hunk")

    os.makedirs(
        os.path.dirname(output_path) or ".",
        exist_ok=True
    )

    total_requests = 0
    skipped_requests = 0

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        for row_idx, (_, row) in enumerate(df.iterrows()):

            # --------------------------------
            # Get candidate hunks
            # --------------------------------

            same_file_candidates = get_same_file_candidates(row)
            other_file_candidates = get_other_file_candidates(row)

            # No candidates -> no LLM request needed
            if (
                not same_file_candidates
                and not other_file_candidates
            ):
                skipped_requests += 1
                continue

            # --------------------------------
            # Build prompt
            # --------------------------------

            messages = build_relevance_messages(
                row,
                same_file_candidates,
                other_file_candidates
            )

            # --------------------------------
            # Build Batch request
            # --------------------------------

            body = {
                "model": MODEL_NAME,
                "messages": messages,
                "response_format": {
                    "type": "json_object"
                }
            }

            if REASONING_EFFORT:
                body["reasoning_effort"] = REASONING_EFFORT
            

            batch_request = {
                "custom_id": (
                    f"context_relevance_"
                    f"{row['patch_id']}_"
                    f"{row_idx}"
                ),
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": body,
            }

            # --------------------------------
            # Write one JSON object per line
            # --------------------------------

            f.write(
                json.dumps(
                    batch_request,
                    ensure_ascii=False
                ) + "\n"
            )

            total_requests += 1

    print(
        f"[DONE] JSONL created: {output_path}"
    )

    print(
        f"[DONE] Total rows: {len(df)}"
    )

    print(
        f"[DONE] Requests created: {total_requests}"
    )

    print(
        f"[DONE] Rows skipped: {skipped_requests}"
    )

    return output_path

In [12]:
build_context_relevance_batch_jsonl(df=df)

[DONE] JSONL created: code_hunks_relevance_batch.jsonl
[DONE] Total rows: 400
[DONE] Requests created: 400
[DONE] Rows skipped: 0


'code_hunks_relevance_batch.jsonl'

getting final dataset for code hunks

In [ ]:
import ast
import json
import re
import pandas as pd


# ============================================================
# Configuration
# ============================================================

RAW_BATCH_FILE = "code_hunks_batch_results.jsonl"
INPUT_DATA_FILE = "../../data/df_n1.csv"
OUTPUT_DATA_FILE = "code_hunks_relevance_results_n2_2.csv"


# ============================================================
# Helpers
# ============================================================

def _parse_hunks_field(value):
    """
    Normalize pr_code_hunks / same_file_code_hunks into
    a Python list of dictionaries.
    """
    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []

    if isinstance(value, list):
        return value

    return []


def get_same_file_candidates(row):
    """
    Reconstruct the same candidate list that was given to the LLM.
    """

    hunks = _parse_hunks_field(row.get("same_file_code_hunks"))

    current_patch = row.get("hunk")
    current_patch = (
        str(current_patch).strip()
        if current_patch is not None
        else None
    )

    candidates = []
    next_id = 1

    for h in hunks:
        if not isinstance(h, dict):
            continue

        patch = h.get("patch")

        if patch is None:
            continue

        # Exclude the target hunk itself
        if current_patch is not None and patch.strip() == current_patch:
            continue

        candidates.append(
            {
                "id": next_id,
                "filename": h.get(
                    "filename",
                    row.get("target_file")
                ),
                "patch": patch,
            }
        )

        next_id += 1

    return candidates


def get_other_file_candidates(row):
    """
    Reconstruct the other-file candidate list that was given
    to the LLM.
    """

    hunks = _parse_hunks_field(row.get("pr_code_hunks"))
    target_file = row.get("target_file")

    candidates = []
    next_id = 1

    for h in hunks:
        if not isinstance(h, dict):
            continue

        filename = h.get("filename")
        patch = h.get("patch")

        if filename is None or patch is None:
            continue

        if filename == target_file:
            continue

        candidates.append(
            {
                "id": next_id,
                "filename": filename,
                "patch": patch,
            }
        )

        next_id += 1

    return candidates


def parse_relevance_output(text):
    """
    Parse the JSON returned by the LLM.

    Expected:
    {
        "relevant_same_file_ids": [1, 3],
        "relevant_other_files_ids": [2]
    }
    """

    empty = {
        "relevant_same_file_ids": [],
        "relevant_other_files_ids": [],
    }

    if text is None:
        return empty

    # Make sure we are working with a string
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()

    # Remove markdown fences if present
    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned
    )

    try:
        data = json.loads(cleaned)

    except json.JSONDecodeError:
        print(
            "[WARN] Could not parse relevance JSON:"
        )
        print(cleaned[:500])

        return empty

    def _as_int_list(value):

        if not isinstance(value, list):
            return []

        result = []

        for v in value:
            try:
                result.append(int(v))
            except (TypeError, ValueError):
                continue

        return result

    return {
        "relevant_same_file_ids": _as_int_list(
            data.get("relevant_same_file_ids", [])
        ),
        "relevant_other_files_ids": _as_int_list(
            data.get("relevant_other_files_ids", [])
        ),
    }


# ============================================================
# Extract text from one Batch API JSONL line
# ============================================================

def extract_batch_output(batch_item):
    """
    Extract the assistant's textual response from a raw
    OpenAI Batch API result.

    Expected Batch API structure is approximately:

    {
        "custom_id": "...",
        "response": {
            "body": {
                "choices": [
                    {
                        "message": {
                            "content": "..."
                        }
                    }
                ]
            }
        }
    }
    """

    try:
        return (
            batch_item["response"]
            ["body"]
            ["choices"][0]
            ["message"]
            ["content"]
        )

    except (KeyError, IndexError, TypeError):
        return None


# ============================================================
# Process one dataframe row using one batch response
# ============================================================

def process_batch_result(row, raw_text):

    # Same candidates that were originally sent to the model
    same_file_candidates = get_same_file_candidates(row)
    other_file_candidates = get_other_file_candidates(row)

    # If there were no candidates, the original API pipeline
    # would have skipped the API call and returned empty lists.
    if not same_file_candidates and not other_file_candidates:
        return {
            "relevant_same_file_code_hunks": [],
            "relevant_different_files_code_hunks": [],
        }

    # Parse model output
    parsed = parse_relevance_output(raw_text)

    # Reconstruct the candidate mappings
    same_map = {
        c["id"]: c
        for c in same_file_candidates
    }

    other_map = {
        c["id"]: c
        for c in other_file_candidates
    }

    # Reconstruct the actual hunk dictionaries
    relevant_same_file_code_hunks = [
        {
            "filename": same_map[i]["filename"],
            "patch": same_map[i]["patch"],
        }
        for i in parsed["relevant_same_file_ids"]
        if i in same_map
    ]

    relevant_different_files_code_hunks = [
        {
            "filename": other_map[i]["filename"],
            "patch": other_map[i]["patch"],
        }
        for i in parsed["relevant_other_files_ids"]
        if i in other_map
    ]

    return {
        "relevant_same_file_code_hunks":
            relevant_same_file_code_hunks,

        "relevant_different_files_code_hunks":
            relevant_different_files_code_hunks,
    }


# ============================================================
# Main post-processing pipeline
# ============================================================

def process_raw_batch_results(
    df,
    raw_batch_file
):

    print("[START] Reading raw batch results...")

    # --------------------------------------------------------
    # Read JSONL
    # --------------------------------------------------------

    batch_results = []

    with open(
        raw_batch_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                batch_results.append(
                    json.loads(line)
                )

            except json.JSONDecodeError as e:
                print(
                    f"[WARN] Invalid JSON on line "
                    f"{line_number}: {e}"
                )

    print(
        f"[INFO] Loaded {len(batch_results)} "
        f"batch results"
    )

    print(
        f"[INFO] Original dataframe has {len(df)} rows"
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # Match results using custom_id when possible.
    # --------------------------------------------------------

    results_by_custom_id = {
        str(item.get("custom_id")): item
        for item in batch_results
        if item.get("custom_id") is not None
    }

    preds = []

    # --------------------------------------------------------
    # Process every dataframe row
    # --------------------------------------------------------

    for index, row in df.iterrows():

        # ----------------------------------------------------
        # Find corresponding batch result
        # ----------------------------------------------------

        # This assumes your custom_id was based on the
        # dataframe row index, e.g. "row_0", "row_1", ...
        custom_id = f"row_{index}"

        batch_item = results_by_custom_id.get(custom_id)

        # ----------------------------------------------------
        # Fallback: positional matching
        # ----------------------------------------------------

        if batch_item is None and index < len(batch_results):

            batch_item = batch_results[index]

        if batch_item is None:

            print(
                f"[WARN] No batch result found "
                f"for dataframe row {index}"
            )

            preds.append(
                {
                    "relevant_same_file_code_hunks": [],
                    "relevant_different_files_code_hunks": [],
                }
            )

            continue

        # ----------------------------------------------------
        # Extract model output
        # ----------------------------------------------------

        raw_text = extract_batch_output(batch_item)

        # ----------------------------------------------------
        # Reconstruct final prediction
        # ----------------------------------------------------

        pred = process_batch_result(
            row,
            raw_text
        )

        preds.append(pred)

    # --------------------------------------------------------
    # Build exactly the same dataframe structure
    # --------------------------------------------------------

    pred_df = pd.DataFrame(preds)

    df = df.reset_index(drop=True)

    result_df = pd.concat(
        [df, pred_df],
        axis=1
    )

    print(
        f"[DONE] Final dataframe shape: "
        f"{result_df.shape}"
    )

    return result_df



[START] Reading raw batch results...
[INFO] Loaded 400 batch results
[INFO] Original dataframe has 400 rows
[DONE] Final dataframe shape: (400, 21)
[SAVED] code_hunks_relevance_results_n2_2.csv


In [ ]:
result_df = process_raw_batch_results(
    df,
    RAW_BATCH_FILE
)
df_relevant_hunks=result_df[['patch_id','relevant_different_files_code_hunks','relevant_same_file_code_hunks']]
df_relevant_hunks.to_csv(
    OUTPUT_DATA_FILE,
    index=False
)


In [9]:
df_relevant_hunks.to_csv(
    "../../data/df_n2_2.csv",)